In [383]:
%run common_setup.ipynb

### This class constructs the pageRank for sources linked by citation counts

-  build the edge list of journal (citer) -> journal (cited)  
-  construct an iGraph from the edge list  
-  run pageRank  
-  for each author, calculate the number of ciations and the number of citations weighted by the pageRank of the citing journal



In [384]:
class ExtractEdgeLists(SetUp):
  
    def __init__(self):
        super().__init__()
        return
  
    def extract_source_edge_list(self):
        print('extract source edge list')

        sql = """
            CREATE OR REPLACE TABLE project.source_edge_list AS
            (
            -- BUILD SOURCE EDGE LISTS FOR PAGERANK
            --**************************************
            WITH
            -->> SELECT source/works in the set
            --**************************************************************************
            sources_CTE AS
                (SELECT DISTINCT id AS work_id,
                        "primary_location.source".id AS source_id,
                    FROM project.raw
                ),
            -->> SELECT citer_cited pairs in the set
            --**************************************  
            citer_cited_CTE AS
                (SELECT *
                FROM
                    (SELECT DISTINCT id AS citer_id,
                        unnest(referenced_works) AS cited_id
                    FROM project.raw
                    )
                WHERE cited_id in (SELECT id FROM project.raw)  
                    -- AND citer_id = 'https://openalex.org/W1000079159' AND cited_id = 'https://openalex.org/W1536446786'
                ),
            -->> SELECT citer works details
            --*****************************
            citer_sources_CTE AS
                (SELECT work_id AS citer_id,
                        source_id AS source_id_citer, 
                        cited_id,
                FROM citer_cited_CTE
                LEFT JOIN sources_CTE
                ON citer_id = work_id
                ),
            
            -->> SELECT citer and cited sources details
            --*****************************
            citer_cited_sources_CTE AS
                (SELECT citer_id,
                        source_id_citer,
                        cited_id,
                        w.source_id AS source_id_cited,
                FROM citer_sources_CTE c
                INNER JOIN sources_CTE w
                ON c.cited_id = w.work_id
                WHERE w.work_id NOT NULL
                ), 
            -->> CONSTRUCT source citation edge list 
            --**************************************
            source_edge_list_CTE AS
                (SELECT DISTINCT source_id_citer AS citer_unit,
                        source_id_cited AS cited_unit,
                        count(citer_id) AS weights,
                FROM citer_cited_sources_CTE
                WHERE source_id_citer != source_id_cited
                GROUP BY ALL
                ) 
            
            -- TESTS
            SELECT *
            FROM source_edge_list_CTE
            )
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.source_edge_list ORDER BY weights DESC").show()
        self.db.sql("SELECT sum(weights) AS sum_of_source_weights FROM project.source_edge_list").show()
        return
  
    def extract_institution_edge_list(self):
        sql = """
            CREATE OR REPLACE TABLE project.institution_edge_list AS
            (
            -- BUILD INSTITUTION EDGE LISTS FOR PAGERANK
            --**************************************
            WITH
            -- -->> SELECT works/institutions in the set
            -- --*************************************
            institutions_CTE AS
                (SELECT DISTINCT work_id,
                        unnest(authorship.institutions).id AS institution_id
                FROM (SELECT DISTINCT id AS work_id,
                                unnest(authorships) AS authorship
                        FROM project.raw
                        )
                ),
            -->> SELECT FILTERED works/institutions in the set 
            -->> (drop institutions with fewer than XX publications in the period)
            --*************************************
            filtered_institutions_CTE AS
                (SELECT i.work_id,
                        i.institution_id
                FROM institutions_CTE i
                INNER JOIN
                    (SELECT institution_id,
                            count(work_id) AS work_counts
                    FROM institutions_CTE
                    GROUP BY ALL
                    ) sub
                USING (institution_id)
                WHERE sub.work_counts > 10*15
                ),
            -->> SELECT citer_cited pairs in the set
            --**************************************  
            citer_cited_CTE AS
                (SELECT *
                FROM
                    (SELECT id AS citer_id,
                            unnest(referenced_works) AS cited_id
                    FROM project.raw
                    ) 
                WHERE cited_id in (SELECT work_id FROM institutions_CTE)
                    -- AND citer_id = 'https://openalex.org/W1000357097' AND cited_id = 'https://openalex.org/W2079631208'
                ),
            -->> SELECT citer institutions
            --*****************************
            citer_institutions_CTE AS
                (SELECT citer_id,
                        institution_id AS citer_institution_id, 
                        cited_id,
                        count(DISTINCT citer_institution_id) OVER (PARTITION BY citer_id, cited_id) AS citer_institutions_count
                FROM citer_cited_CTE
                INNER JOIN filtered_institutions_CTE
                ON citer_id = work_id
                ),
            
            -->> SELECT cited instutions
            --*****************************
            cited_institutions_CTE AS
                (SELECT DISTINCT citer_id,
                        cited_id,
                        w.institution_id AS cited_institution_id,
                        count(DISTINCT cited_institution_id) OVER (PARTITION BY citer_id, cited_id) AS cited_institutions_count
                FROM citer_cited_CTE c
                INNER JOIN filtered_institutions_CTE w
                ON c.cited_id = w.work_id
                ), 
            -->> JOIN citer and cited institutions 
            --**************************************
            citer_cited_institutions_CTE AS
                (SELECT r.citer_id,
                        r.citer_institution_id,
                        r.citer_institutions_count,
                        d.cited_id,
                        d.cited_institution_id,
                        d.cited_institutions_count
                FROM citer_institutions_CTE r
                INNER JOIN cited_institutions_CTE d
                ON r.citer_id = d.citer_id AND r.cited_id = d.cited_id
                ),
            
            -->> CONSTRUCT source citation edge list 
            --**************************************
            institution_edge_list_CTE AS
                (SELECT DISTINCT citer_institution_id AS citer_unit,
                        cited_institution_id AS cited_unit,
                        sum(1.0/(citer_institutions_count * cited_institutions_count)) AS weights
                FROM citer_cited_institutions_CTE
                WHERE citer_institution_id != cited_institution_id
                GROUP BY ALL
                ) 

            -- TEST
            SELECT *
            FROM institution_edge_list_CTE
            )
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.institution_edge_list ORDER BY weights DESC").show()
        self.db.sql("SELECT sum(weights) AS sum_of_institution_weights FROM project.institution_edge_list").show()
        return
    
    def both_edge_list(self):
        self.db.sql("SELECT sum(weights) AS source_weight_sum FROM project.source_edge_list").show()
        self.db.sql("SELECT sum(weights) AS institution_weight_sum FROM project.institution_edge_list").show()
        sql = """
            CREATE OR REPLACE TABLE project.both_edge_list AS
                (SELECT * FROM project.institution_edge_list
                UNION
                SELECT * FROM project.source_edge_list
                )
            """
        self.db.sql(sql)
        return
    
    def article_vector(self):
        print('article vector')
        sql = """  
            -- ARTICLE VECTORS=======================
            CREATE OR REPLACE TABLE project.article_vectors AS
                (WITH 
                vertex_labler_CTE AS
                    (SELECT work_id,
                            source_id,
                            source_name,

                            unnest(authorship.institutions).id AS institution_id,
                            unnest(authorship.institutions).display_name AS institution_name,
                    FROM
                        (SELECT id AS work_id,
                                "primary_location.source".id AS source_id,
                                "primary_location.source".display_name AS source_name,
                                unnest(authorships) AS authorship,
                        FROM project.raw
                        )
                    )

                SELECT count(DISTINCT work_id) OVER (PARTITION BY source_id) AS work_count,
                        source_id AS unit_id,
                        source_name AS unit_name,
                FROM vertex_labler_CTE
                UNION
                SELECT *
                    FROM
                        (SELECT count(DISTINCT work_id) OVER (PARTITION BY institution_id) AS work_count,
                                institution_id AS unit_id,
                                institution_name AS unit_name,
                            FROM vertex_labler_CTE
                        )
                    WHERE work_count > 10*15
                )
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM project.article_vectors").show()
        return

In [385]:
class PageRank(SetUp):

    def __init__(self):
        super().__init__()
        return

    def vertex_labels(self):

        sql = """
            -- VERTEX LABLES
            -- ========================================
            WITH 
            vertex_labler_CTE AS
                (SELECT source_id,
                        source_name,
                        unnest(authorship.institutions).id AS institution_id,
                        unnest(authorship.institutions).display_name AS institution_name,
                FROM
                    (SELECT id AS work_id,
                        "primary_location.source".id AS source_id,
                        "primary_location.source".display_name AS source_name,
                        unnest(authorships) AS authorship,
                    FROM project.raw
                    )
                )

            SELECT source_id AS unit_id,
                    source_name AS unit_name,
            FROM vertex_labler_CTE
            UNION
            (SELECT institution_id AS unit_id,
                    institution_name AS unit_name,
                FROM vertex_labler_CTE)
            """
        df = self.db.sql(sql).df()
        print(df.head())
        self.vertex_labels = dict(zip(df.unit_id, df.unit_name))
        return
    
    def article_vectors(self):
        df = self.db.sql("SELECT * FROM project.article_vectors").df()
        print(df.head())
        countS = df[['/S' in x for x in df.unit_id]].work_count.sum()
        countI = df[['/I' in x for x in df.unit_id]].work_count.sum()
        print(f'{countS = } {countI = }')

        return

    def _extract_edge_list(self):
        kind = self.kind
        df = self.db.sql(f"SELECT citer_unit, cited_unit, weights FROM project.{kind}_edge_list").df().\
            sort_values('weights', ascending=False).reset_index(drop=True)
        print(f'{kind = } {df.weights.max() = }')
        return df
    
    def construct_graph(self, kind=None):
        self.kind = kind
        df_edges = self._extract_edge_list()    
        vertices = set(df_edges.citer_unit.tolist() + df_edges.cited_unit.tolist())
        vs = pd.DataFrame(data=list(vertices), columns=['id'])
        vs['label'] = [self.vertex_labels.get(id) for id in vs.id]
        print(f'{vs['id'].nunique() = } {df_edges.shape = }\n{df_edges.head()}\n{df_edges.tail()}')
        print(f'GRAPH NODES - Vertex count for {self.kind = } {vs.shape = }\n{vs.head()}')
        self.g = ig.Graph.DataFrame(df_edges, directed=True, use_vids=False, vertices=vs)
        summary = ig.summary(self.g, verbosity=1, width=256, edge_list_format='auto', max_rows=2, print_graph_attributes=True, 
                                          print_vertex_attributes=True, print_edge_attributes=True, full=False)
        print(f'*** SUMMARY OF self.g\n{summary}')
        return
    
    def run_pagerank(self):
        print('pageRank')
        damping = 0.5  # if self.kind == 'both' else 0.85
        print(f'>> RUN pagerank with {damping = } for {self.kind = }')
        ranks = self.g.pagerank(damping=damping)
        print(f'>> CHECK pageRank  - should sum to unity {sum(ranks) = } then scaled to a sum of 100 as in "eigenfactor" score')
        sum_ranks = sum(ranks)
        scaled_ranks = [100.0 * r / sum_ranks for r in ranks]
        pagerank = pd.DataFrame({
                                'pageRank': scaled_ranks,
                                'citer': self.g.vs['name'],
                                'label': self.g.vs['label'],
                                'in_degree': self.g.strength(mode='in', weights='weights'),
                                'out_degree': self.g.strength(mode='out', weights='weights')
                                }).sort_values('pageRank', ascending=False)
        total_out_degree = pagerank.out_degree.sum()
        pagerank['influence'] = pagerank['pageRank'] * (total_out_degree/pagerank.out_degree) . replace(0, pd.NA)
        self.db.sql(f"CREATE OR REPLACE TABLE project.pagerank_{self.kind} AS SELECT * FROM pagerank")
        return
    
    def report_pagerank(self):
        kind = self.kind
        df = (
            self.db.sql(f"SELECT * FROM project.pagerank_{kind}")
            .df()
            .sort_values('pageRank', ascending=False)
            .reset_index(drop=True)
        )
        print(f"Pagerank for {kind!r}: {df.shape}\n{df.head()}")
        print(f"Sum of out_degrees: {df['out_degree'].sum()}")
        print(f"Sum of pageranks: {df['pageRank'].sum()}")
        sum_weights = self.db.sql(f"SELECT SUM(weights) AS total_weights FROM project.{kind}_edge_list").df().iloc[0, 0]
        print(f"Sum of edge weights: {sum_weights}")
        return

In [386]:
class Plotters(SetUp):

    def __init__(self):
        super().__init__()
        return

    def plot_pagerank(self):
        hold = []
        for kind in ['source', 'institution', 'both']:
            temp = self.db.sql(f"SELECT * FROM project.pagerank_{kind}").df().sort_values('pageRank', ascending=True).reset_index(drop=True).reset_index(drop=False)
            temp['panel'] = kind
            hold.append(temp)
        df = pd.concat(hold, axis=0)
        df = df.melt(id_vars=['out_degree', 'panel'], value_vars=['pageRank', 'influence'], var_name='measure', value_name='Value')
        print(f'{df.shape = }\n{df.head()}\n{df.info()}')
        grouped = df.groupby(['panel', 'measure']).agg({'out_degree': ['min', 'max'], 'Value': ['min', 'max']})
        print(grouped)
        g = sns.relplot(df, x='out_degree', y='Value', hue='measure', col='panel', row='measure', kind='scatter')
        for ax in g.axes.flat:
            ax.set_xlim((10, 50000))
            ax.set_ylim((0.01, 100))
            ax.set_xscale('log')
            ax.set_yscale('log')
        plt.suptitle("PageRank and Influence versus out-degree (i.e. emitted citations)", fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
        return

    def plot_model(self):

        df = self.summary[['author_id', 'citations_endogenous', 
                           'reputation_sources', 'reputation_institutions', 'reputation_both', 
                           'hca_endogenous', 'hca_total', '2yr_mean_citedness', 'h_index', 'group']]
        print(f'{df.shape = }\n{df.head()}\n{df.info()}')
        df['pointsize'] = [0.01 if s == 'X' else 5.0 if s == 'C' else 15 for s in df['group']]
        # df = df.sort_values(['group', 'pointsize'], ascending=[False, True])
        hold = []
        for kind in ['source', 'institution', 'both']:
            temp = df.copy()
            temp['reputation'] = temp[f'reputation_{kind}']
            temp['panel'] = kind
            hold.append(temp)
        df_in = pd.concat(hold, axis=0)
        self._plot_reputations(df_in=df_in)
        self._plot_hca(df_in=df_in)

        # df = df[df.group != 'X']
        # df = df[['citer_count_weighted', 'citer_count', 'ratio', 'author_name', 'group']].sort_values('ratio', ascending=False).reset_index(drop=True)
        # df.to_csv(f'../DATA/weighted_citations_{kind}.csv')

        return

    def _plot_reputations(self, df_in=None):

        print(df_in.info())
        df = df_in.melt(id_vars=['panel', 'author_id', 'citations_endogenous', 'group', 'pointsize'], 
                        value_vars=['reputation'], 
                        value_name='Value', 
                        var_name='measure')
        print(f'{df.shape = }\n{df.head()}')      
        g = sns.relplot(df, x='citations_endogenous', y='Value', hue='group', size='pointsize', col='panel', kind='scatter', alpha=0.5)
        for ax in g.axes.flat:
            ax.set_xscale('log')
            ax.set_yscale('log')
        g.set(xlabel='Endogenous citation count per author', ylabel="Author's reputation score")
        plt.suptitle("Reputation versus citations", fontsize=16)
        sns.move_legend(g, 'upper right')
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.show()
        return

    def _plot_hca(self, df_in=None):

        for kind in ['hca_total', 'hca_endogenous', 'h_index', '2yr_mean_citedness']:

            df = df_in.melt(id_vars=['panel', 'author_id', kind, 'group', 'pointsize'], 
                            value_vars=['reputation'], 
                            value_name='Value', 
                            var_name='measure')
            print(f'{df.shape = }\n{df.head()}')    
            g = sns.relplot(df, x=kind, y='Value', hue='group', size='pointsize', col='panel', kind='scatter', alpha=0.5)
            for ax in g.axes.flat:
                ax.set_xscale('log')
                ax.set_yscale('log')
            g.set(xlabel=f"Author's {kind.replace('_', ' ').upper()}", ylabel="Author's reputation score")
            plt.suptitle(f"{kind.replace('_', ' ').upper()} versus Reputation", fontsize=16)
            sns.move_legend(g, 'upper right')
            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.show()
        return
        
    def extract_data(self):

        self.summary = self.db.sql("SELECT * FROM project.citation_summary").df()\
            [['author_id', 'author_name', 'works_count_endogenous', 'citations_endogenous', 'hca_total', 
             'hca_endogenous', 'works_count_total', 'cited_by_count', '2yr_mean_citedness', 'h_index']]

        for kind in ['sources', 'institutions', 'both']:
            temp = self.db.sql(f"SELECT * FROM project.weighted_citations_{kind}").df().sort_values('citer_count_weighted', ascending=False)
            temp['reputation'] = temp.citer_count_weighted/temp.citer_count
            dd = dict(zip(temp.author_id, temp.reputation))
            self.summary[f'reputation_{kind}'] = [dd.get(a) for a in self.summary.author_id]

        sample = self.db.sql("SELECT * FROM project.sample_names").df()
        dd_group = dict(zip(sample.author_id, sample.Group))
        self.summary['group'] = [dd_group.get(aid, 'X') for aid in self.summary.author_id]
        
        summary = self.summary.rename(columns={"2yr_mean_citedness": "citedness"})
        print(f'{self.summary.shape = }\n{self.summary.head()}')
        self.db.sql("SELECT * FROM summary").show()
        self.db.sql("CREATE OR REPLACE TABLE project.summary AS (SELECT * FROM summary)")
        return

In [387]:
def main():

    exel = ExtractEdgeLists()
    # exel.extract_source_edge_list()
    # exel.extract_institution_edge_list()
    # exel.both_edge_list()
    # exel.article_vector()
    exel.db.close()

    pr = PageRank()
    pr.vertex_labels()
    pr.article_vectors()
    # for kind in ['source', 'institution', 'both']:
    #     pr.construct_graph(kind=kind)
    #     pr.run_pagerank()
    #     pr.report_pagerank()
    # pr.db.close()

    # p = Plotters()
    # # p.extract_data()
    # p.plot_pagerank()
    # # p.plot_model()
    # p.db.close()

In [388]:
if __name__ == "__main__":
    main()
    print("DONE!")

InternalException: INTERNAL Error: Failed to load metadata pointer (id 590, idx 0, ptr 590)


Stack Trace:

/home/lc/Projects/EconomicsBusiness/.venv/lib/python3.12/site-packages/duckdb/duckdb.cpython-312-x86_64-linux-gnu.so(_ZN6duckdb9Exception6ToJSONENS_13ExceptionTypeERKSs+0x53) [0x7203ca15a5f3]
/home/lc/Projects/EconomicsBusiness/.venv/lib/python3.12/site-packages/duckdb/duckdb.cpython-312-x86_64-linux-gnu.so(_ZN6duckdb9ExceptionC1ENS_13ExceptionTypeERKSs+0x16) [0x7203ca15a626]
/home/lc/Projects/EconomicsBusiness/.venv/lib/python3.12/site-packages/duckdb/duckdb.cpython-312-x86_64-linux-gnu.so(_ZN6duckdb17InternalExceptionC1ERKSs+0x11) [0x7203ca15cc31]
/home/lc/Projects/EconomicsBusiness/.venv/lib/python3.12/site-packages/duckdb/duckdb.cpython-312-x86_64-linux-gnu.so(_ZN6duckdb17InternalExceptionC2IJljmEEERKSsDpT_+0x187) [0x7203cada1c67]
/home/lc/Projects/EconomicsBusiness/.venv/lib/python3.12/site-packages/duckdb/duckdb.cpython-312-x86_64-linux-gnu.so(+0xad9ecc) [0x7203c94d9ecc]
/h